# Sionna RT — USD / Omniverse Scene Builder
## Build a radio scene directly from a USD (NVIDIA Omniverse) stage — no OpenStreetMap

This is the USD/Omniverse counterpart to `sionna019_scene_builder_london.ipynb`.
Instead of downloading OSM building/road/vegetation footprints and extruding them,
it reads real mesh geometry + materials directly out of a `.usd`/`.usda`/`.usdc`
stage (e.g. exported from Omniverse, or imported there from another tool) and
converts it into the same `mat_plys` → `scene.xml` pipeline the OSM builder uses,
so the output is a drop-in for `sionna2_915mhz_dem_simulation_london.ipynb` /
`sionna018_neural_calibration_london.ipynb`.

**Status: untested against a real USD file.** `DEMO_MODE=True` (CELL 0) generates
a small synthetic USD-like scene in memory so every cell is runnable today without
`usd-core` installed and without an Omniverse export to test against. Once you
have a real `.usd` file, set `DEMO_MODE=False` and `USD_FILE` to its path.

---

## Build Sequence
1. CELL 0 — Configuration (paths, `PROJECTION_CRS`, `DEMO_MODE`)
2. CELL 1 — Imports (`pxr`/usd-core, optional `trimesh`)
3. CELL 2 — Load USD stage → flatten mesh prims to world-space verts/faces
4. CELL 3 — Map USD material/shader names → ITU-R P.2040-2 material names
5. CELL 4 — Write per-mesh PLYs (`meshes/`)
6. CELL 5 — Write `scene.xml` (Sionna 0.19 Mitsuba format — same writer as the OSM builder)
7. CELL 6 — 2D top-down preview of extracted geometry
8. CELL 7 — Load into Sionna RT and sanity-check materials/bbox

— Scene is then ready for `sionna2_915mhz_dem_simulation_london.ipynb` /
`sionna018_neural_calibration_london.ipynb` (point `SCENE_XML` at the output).


## CELL 0 — Configuration

In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this block only)
# ============================================================
import os

SCENARIO_NAME = 'london_omniverse_usd'
CITY_NAME     = 'London'

# ── Demo mode ─────────────────────────────────────────────────────────────────
# True  -> build a small synthetic in-memory scene (no usd-core, no real file
#          needed) so the rest of the notebook is runnable today.
# False -> load USD_FILE for real via pxr/usd-core.
DEMO_MODE = True
USD_FILE  = '/path/to/your/omniverse_scene.usd'   # only used when DEMO_MODE=False

BASE_DIR  = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', SCENARIO_NAME))
SCENE_DIR = os.path.join(BASE_DIR, 'scene_usd')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

# ── Coordinate system ─────────────────────────────────────────────────────────
# USD scenes are usually authored in a local Cartesian frame (metres) already,
# with no inherent GPS anchor -- unlike OSM, which is always WGS84-derived.
# If your USD stage carries real-world georeference metadata, set
# USD_HAS_GEOREFERENCE=True and fill in the anchor; otherwise the scene origin
# is just wherever the USD stage's own (0,0,0) is, and you must align it to
# your TX/RX GPS positions manually (e.g. by placing TX at a known USD prim).
USD_HAS_GEOREFERENCE = True     # True -> DEM terrain drape (CELL 2b) is active by default
ANCHOR_LON = -0.13399    # only used if USD_HAS_GEOREFERENCE=True
ANCHOR_LAT =  51.5305

# Fixed scene bbox (WGS84) -- same area as sionna019_scene_builder_london.ipynb's
# SCENE_WEST/EAST/SOUTH/NORTH, so the DEM terrain and OSM clutter (CELL 2b/2c) cover
# the same real-world extent as that notebook, instead of just hugging the USD buildings.
USE_FIXED_SCENE_BBOX = True
SCENE_WEST  = -0.231017
SCENE_EAST  = -0.036963
SCENE_SOUTH = 51.47014
SCENE_NORTH = 51.59086

# ── Real elevation (EA LiDAR DTM) — drape USD buildings onto real terrain ──
# Requires USD_HAS_GEOREFERENCE=True (ANCHOR_LON/ANCHOR_LAT must be the real
# WGS84 location of the USD stage's local origin (0,0,0)) -- otherwise there
# is no way to look up a real-world elevation for the USD scene's (x, y).
USE_DEM_TERRAIN  = True     # False -> flat terrain (old behaviour)
TERRAIN_GRID_N   = 200      # terrain.ply resolution (N x N grid)
TERRAIN_PAD_M    = 200.0    # terrain extent beyond the USD scene bbox (m)
EA_DTM_TIFF      = os.path.join(BASE_DIR, 'dem.tif')

# ── OSM real-world clutter (roads, vegetation, water) around the USD buildings ──
# USD scenes typically only contain the buildings/objects the artist modeled --
# this fills the gaps with real OSM features for the same bbox, draped onto the
# same DEM terrain as the buildings (CELL 2b). Requires USD_HAS_GEOREFERENCE=True
# (needs ANCHOR_LON/ANCHOR_LAT to know where on Earth the USD scene actually is).
INCLUDE_OSM_ROADS      = True
INCLUDE_OSM_VEGETATION = True
INCLUDE_OSM_WATER      = True
OSM_BBOX_PAD_M    = 100.0   # extra margin (m) around the USD scene bbox for the OSM query
ROAD_HEIGHT_M     = 0.05    # road mesh thickness
VEGETATION_HEIGHT_M = 8.0   # default tree/canopy height (m) when OSM has no height tag
WATER_HEIGHT_M    = 0.10    # water mesh thickness

# Must match PROJECTION_CRS in sionna2_915mhz_dem_simulation_london.ipynb /
# sionna019_scene_builder_london.ipynb / sionna018_neural_calibration_london.ipynb
PROJECTION_CRS  = 'bng'    # 'bng' | 'utm30n'
_PROJECTION_EPSG_MAP = {'bng': 27700, 'utm30n': 32630}
UTM_EPSG = _PROJECTION_EPSG_MAP.get(PROJECTION_CRS, 27700)

FREQUENCY_HZ = 915.95e6

print(f'Scenario   : {SCENARIO_NAME}')
print(f'Demo mode  : {DEMO_MODE}')
print(f'USD file   : {USD_FILE if not DEMO_MODE else "(none -- synthetic demo scene)"}')
print(f'Scene dir  : {SCENE_DIR}')
print(f'Projection : {PROJECTION_CRS}  ->  EPSG:{UTM_EPSG}')
print(f'DEM terrain: {USE_DEM_TERRAIN and USD_HAS_GEOREFERENCE} ' + ('' if USD_HAS_GEOREFERENCE else '(needs USD_HAS_GEOREFERENCE=True)'))


## CELL 1 — Imports

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, json
import numpy as np

try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
print(f'trimesh  : {"OK" if _HAS_TRIMESH else "not available -- falls back to ASCII PLY writer"}')

_HAS_USD = False
if not DEMO_MODE:
    try:
        from pxr import Usd, UsdGeom, UsdShade, Gf
        _HAS_USD = True
        print('pxr (usd-core) : OK')
    except ImportError as _e:
        print(f'pxr (usd-core) : NOT AVAILABLE -- {_e}')
        print('  Install with:  pip install usd-core')
        print('  Falling back to DEMO_MODE behaviour for this run.')
else:
    print('pxr (usd-core) : skipped (DEMO_MODE=True)')


## CELL 2 — Load USD Stage → Flatten Mesh Prims
Real path: opens the stage with `Usd.Stage.Open`, walks every `UsdGeom.Mesh` prim,
applies its computed world transform (`UsdGeom.XformCache`), and triangulates
`faceVertexCounts`/`faceVertexIndices` (USD meshes can have n-gon faces, not just
triangles) into `(verts, tri_faces, material_name)` per prim.

Demo path: builds 4 synthetic box prims (3 "buildings" + 1 "ground slab") with
material names chosen to exercise the ITU mapping in CELL 3, standing in for
what `UsdShadeMaterialBindingAPI` would have returned from a real stage.

In [ ]:
# ============================================================
# CELL 2 — LOAD USD STAGE → WORLD-SPACE MESH PRIMS
# ============================================================
# Output: usd_meshes = [{'name': str, 'verts': (N,3) float32,
#                         'faces': (M,3) int32, 'material': str}, ...]

def _box_mesh(cx, cy, cz, sx, sy, sz):
    """Axis-aligned box centred at (cx,cy,cz), half-extents (sx,sy,sz)/2."""
    x0, x1 = cx - sx/2, cx + sx/2
    y0, y1 = cy - sy/2, cy + sy/2
    z0, z1 = cz, cz + sz
    v = np.array([
        [x0,y0,z0],[x1,y0,z0],[x1,y1,z0],[x0,y1,z0],  # bottom
        [x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1],  # top
    ], dtype=np.float32)
    f = np.array([
        [0,1,2],[0,2,3],          # bottom
        [4,6,5],[4,7,6],          # top
        [0,4,5],[0,5,1],          # sides
        [1,5,6],[1,6,2],
        [2,6,7],[2,7,3],
        [3,7,4],[3,4,0],
    ], dtype=np.int32)
    return v, f

usd_meshes = []

if DEMO_MODE or not _HAS_USD:
    print('Building synthetic demo scene (3 buildings + ground slab) ...')
    v, f = _box_mesh(0, 0, 0, 400, 300, 0.2);      usd_meshes.append({'name': 'ground',     'verts': v, 'faces': f, 'material': 'Concrete_Ground'})
    v, f = _box_mesh(-80, 60, 0, 40, 30, 22.0);    usd_meshes.append({'name': 'building_A', 'verts': v, 'faces': f, 'material': 'Brick_Facade'})
    v, f = _box_mesh(60, -40, 0, 50, 50, 35.0);    usd_meshes.append({'name': 'building_B', 'verts': v, 'faces': f, 'material': 'Glass_Curtain_Wall'})
    v, f = _box_mesh(20, 90, 0, 30, 20, 12.0);     usd_meshes.append({'name': 'building_C', 'verts': v, 'faces': f, 'material': 'Metal_Cladding'})
    print(f'  {len(usd_meshes)} synthetic prims created (demo data, NOT a real scene)')
else:
    print(f'Opening USD stage: {USD_FILE}')
    stage = Usd.Stage.Open(USD_FILE)
    assert stage is not None, f'Could not open USD stage: {USD_FILE}'
    xform_cache = UsdGeom.XformCache()

    def _triangulate(face_counts, face_indices):
        tris = []
        idx = 0
        for n in face_counts:
            poly = face_indices[idx: idx + n]
            for k in range(1, n - 1):
                tris.append([poly[0], poly[k], poly[k + 1]])
            idx += n
        return np.asarray(tris, dtype=np.int32)

    def _bound_material_name(prim):
        try:
            rel = UsdShade.MaterialBindingAPI(prim).ComputeBoundMaterial()
            mat = rel[0] if isinstance(rel, tuple) else rel
            if mat and mat.GetPrim().IsValid():
                return mat.GetPrim().GetName()
        except Exception:
            pass
        return 'DEFAULT'

    for prim in stage.Traverse():
        if not prim.IsA(UsdGeom.Mesh):
            continue
        mesh = UsdGeom.Mesh(prim)
        pts = np.asarray(mesh.GetPointsAttr().Get(), dtype=np.float64)
        if pts is None or len(pts) == 0:
            continue
        counts = mesh.GetFaceVertexCountsAttr().Get()
        idxs   = mesh.GetFaceVertexIndicesAttr().Get()
        faces  = _triangulate(counts, idxs)

        world = xform_cache.GetLocalToWorldTransform(prim)
        world_np = np.array(world).reshape(4, 4)
        pts_h = np.hstack([pts, np.ones((len(pts), 1))])
        pts_world = (pts_h @ world_np)[:, :3].astype(np.float32)

        usd_meshes.append({
            'name': prim.GetName(),
            'verts': pts_world,
            'faces': faces,
            'material': _bound_material_name(prim),
        })
    print(f'  {len(usd_meshes)} mesh prims extracted from stage')

for m in usd_meshes:
    print(f"  {m['name']:<16} verts={len(m['verts']):>5}  faces={len(m['faces']):>5}  material={m['material']}")


## CELL 2b — EA LiDAR DTM Terrain + Drape USD Buildings onto Real Elevation
Downloads the Environment Agency 1 m Composite DTM (bare-earth terrain) for
the bbox around `(ANCHOR_LON, ANCHOR_LAT)`, builds a real `terrain.ply` from
it (same grid-sampling approach as `sionna019_scene_builder_london.ipynb`
CELL 3), then **drapes** every USD mesh by shifting its vertices' Z by the
real terrain height at its footprint centroid — since USD meshes are
typically authored on a local flat (z=0) ground plane, not on real elevation.

Skips automatically (flat terrain, old behaviour) if `USE_DEM_TERRAIN=False`
or `USD_HAS_GEOREFERENCE=False` (no real-world anchor to look up elevation
against).


In [ ]:
# ============================================================
# CELL 2b — EA LIDAR DTM TERRAIN + DRAPE
# ============================================================
_dem_ok = False

if not (USE_DEM_TERRAIN and USD_HAS_GEOREFERENCE):
    print('Skipping DEM terrain -- USE_DEM_TERRAIN/USD_HAS_GEOREFERENCE not both True.')
    print('Buildings stay on the flat z=0 plane they were authored on.')
else:
    import requests
    import numpy as np
    from pyproj import Transformer

    print('=' * 60)
    print('CELL 2b -- EA LiDAR DTM download + terrain drape')
    print('=' * 60)

    # USD local (x, y) in metres around the anchor -> real-world BNG/UTM (e, n)
    _wgs_to_bng = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
    _bng_to_wgs = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
    _anchor_e, _anchor_n = _wgs_to_bng.transform(ANCHOR_LON, ANCHOR_LAT)

    # Scene extent in USD local metres, from the meshes already loaded in CELL 2
    _all_xy = np.concatenate([m['verts'][:, :2] for m in usd_meshes], axis=0)
    _x_min, _x_max = float(_all_xy[:, 0].min()), float(_all_xy[:, 0].max())
    _y_min, _y_max = float(_all_xy[:, 1].min()), float(_all_xy[:, 1].max())

    if globals().get('USE_FIXED_SCENE_BBOX', False):
        # Same fixed area as sionna019_scene_builder_london.ipynb's SCENE_WEST/
        # EAST/SOUTH/NORTH, converted to BNG, instead of just padding the USD bbox.
        _e_min, _n_min = _wgs_to_bng.transform(SCENE_WEST, SCENE_SOUTH)
        _e_max, _n_max = _wgs_to_bng.transform(SCENE_EAST, SCENE_NORTH)
    else:
        _e_min = _anchor_e + _x_min - TERRAIN_PAD_M
        _e_max = _anchor_e + _x_max + TERRAIN_PAD_M
        _n_min = _anchor_n + _y_min - TERRAIN_PAD_M
        _n_max = _anchor_n + _y_max + TERRAIN_PAD_M
    print(f'Scene extent (local) : x=[{_x_min:.0f},{_x_max:.0f}]  y=[{_y_min:.0f},{_y_max:.0f}]')
    print(f'Real-world bbox (EPSG:{UTM_EPSG}) : E=[{_e_min:.0f},{_e_max:.0f}]  N=[{_n_min:.0f},{_n_max:.0f}]')

    _WCS_ENDPOINT = 'https://environment.data.gov.uk/spatialdata/lidar-composite-digital-terrain-model-dtm-1m/wcs'

    def _wcs_get_coverage_id(default='13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m'):
        """Auto-detect the real CoverageId via GetCapabilities -- the EA WCS
        has renamed/changed this id before, which surfaces as a 404 or a
        500 (server throws on an unrecognised id) on GetCoverage even
        though the service itself is up."""
        try:
            _cap = requests.get(_WCS_ENDPOINT, params={
                'SERVICE': 'WCS', 'VERSION': '2.0.1', 'REQUEST': 'GetCapabilities'},
                timeout=30)
            _cap.raise_for_status()
            import re as _re
            _ids = _re.findall(r'<(?:wcs:)?CoverageId>([^<]+)</(?:wcs:)?CoverageId>', _cap.text)
            print(f'  GetCapabilities advertises {len(_ids)} coverage(s): {_ids[:10]}'
                  f'{" ..." if len(_ids) > 10 else ""}')
            if _ids:
                _dtm_ids = [i for i in _ids if 'dtm' in i.lower()] or _ids
                if default not in _dtm_ids:
                    print(f'  CoverageId "{default}" not advertised; '
                          f'using "{_dtm_ids[0]}" instead.')
                return _dtm_ids[0] if default not in _dtm_ids else default
        except Exception as _e:
            print(f'  GetCapabilities probe failed: {_e} -- keeping default CoverageId.')
        return default

    if os.path.exists(EA_DTM_TIFF):
        print(f'Already downloaded: {EA_DTM_TIFF}')
    else:
        def _wcs_url(coverage_id):
            return (
                _WCS_ENDPOINT +
                '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
                f'&COVERAGEID={coverage_id}'
                f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_e_min:.0f},{_e_max:.0f})'
                f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_n_min:.0f},{_n_max:.0f})'
                '&FORMAT=image/tiff'
            )

        print('Downloading EA LiDAR DTM ...')
        _coverage_ids_to_try = ['13787b9a-26a4-4775-8523-806d13af58fc__Lidar_Composite_Elevation_DTM_1m']
        try:
            _r = requests.get(_wcs_url(_coverage_ids_to_try[0]), timeout=120)
            if _r.status_code in (404, 500):
                # CoverageId mismatch is the most common cause of a 404/500 on
                # an otherwise-valid, small, in-coverage bbox (a 500 means the
                # server itself threw on an unrecognised id rather than
                # cleanly 404ing) -- re-probe via GetCapabilities and retry
                # once with the discovered id.
                print(f'  {_r.status_code} on CoverageId "{_coverage_ids_to_try[0]}" -- probing GetCapabilities ...')
                _alt_id = _wcs_get_coverage_id(_coverage_ids_to_try[0])
                if _alt_id != _coverage_ids_to_try[0]:
                    _coverage_ids_to_try.append(_alt_id)
                    _r = requests.get(_wcs_url(_alt_id), timeout=120)
            _r.raise_for_status()
            os.makedirs(os.path.dirname(EA_DTM_TIFF), exist_ok=True)
            with open(EA_DTM_TIFF, 'wb') as _f:
                _f.write(_r.content)
            print(f'  saved {os.path.getsize(EA_DTM_TIFF) // 1024} KB -> {EA_DTM_TIFF}')
        except Exception as _e:
            print(f'  WCS download failed: {_e}')
            print('  Tried CoverageId(s): ' + ', '.join(_coverage_ids_to_try))
            print('  If this persists, the WCS service may be down or your bbox is offshore/'
                  'outside England. Manual fallback: download the 1m DTM tile for your area from')
            print('  https://environment.data.gov.uk/DefraDataDownload/?Mode=survey '
                  '(LIDAR Composite DTM, 1m), then point EA_DTM_TIFF (CELL 0) at the .tif directly.')

    if os.path.exists(EA_DTM_TIFF):
        try:
            import rasterio
            with rasterio.open(EA_DTM_TIFF) as _ds:
                _dem_arr = _ds.read(1).astype(np.float32)
                _dem_transform = _ds.transform
            _rows, _cols = _dem_arr.shape
            _dem_es = np.array([_dem_transform * (c + 0.5, 0) for c in range(_cols)])[:, 0]
            _dem_ns = np.array([_dem_transform * (0, r + 0.5) for r in range(_rows)])[:, 1]

            from scipy.interpolate import RegularGridInterpolator
            _dem_interp = RegularGridInterpolator(
                (_dem_ns[::-1], _dem_es), _dem_arr[::-1, :],
                bounds_error=False, fill_value=None)

            def local_z(x, y):
                """Real terrain height (m) at USD local (x, y), relative to anchor elevation."""
                e, n = _anchor_e + x, _anchor_n + y
                return float(_dem_interp([[n, e]])[0]) - _origin_elev_asl

            _origin_elev_asl = float(_dem_interp([[_anchor_n, _anchor_e]])[0])
            print(f'Anchor elevation : {_origin_elev_asl:.2f} m ASL')

            # ── Build real terrain.ply over the scene extent ──────────────
            _xs = np.linspace(_x_min - TERRAIN_PAD_M, _x_max + TERRAIN_PAD_M, TERRAIN_GRID_N, dtype=np.float32)
            _ys = np.linspace(_y_min - TERRAIN_PAD_M, _y_max + TERRAIN_PAD_M, TERRAIN_GRID_N, dtype=np.float32)
            _XX, _YY = np.meshgrid(_xs, _ys)
            _ZZ = np.array([[local_z(float(xv), float(yv)) for xv in _xs] for yv in _ys], dtype=np.float32)
            _tv = np.stack([_XX, _YY, _ZZ], axis=-1).reshape(-1, 3)
            _tf = []
            for _r in range(TERRAIN_GRID_N - 1):
                for _c in range(TERRAIN_GRID_N - 1):
                    _i0 = _r * TERRAIN_GRID_N + _c
                    _i1 = _i0 + 1
                    _i2 = _i0 + TERRAIN_GRID_N
                    _i3 = _i2 + 1
                    _tf.append([_i0, _i2, _i1]); _tf.append([_i1, _i2, _i3])
            _tf = np.asarray(_tf, dtype=np.int32)

            # Replace any synthetic 'ground' mesh with the real DEM terrain mesh
            usd_meshes = [m for m in usd_meshes if m['name'] != 'ground']
            usd_meshes.append({'name': 'dem_terrain', 'verts': _tv, 'faces': _tf,
                                'material': 'EA_LiDAR_Terrain'})

            # ── Drape every building mesh: shift Z by terrain height at its footprint centroid ──
            for m in usd_meshes:
                if m['name'] == 'dem_terrain':
                    continue
                cx, cy = float(m['verts'][:, 0].mean()), float(m['verts'][:, 1].mean())
                dz = local_z(cx, cy)
                m['verts'] = m['verts'].copy()
                m['verts'][:, 2] += dz
                print(f"  draped {m['name']:<16} +{dz:6.2f} m  (terrain height at footprint centroid)")

            _dem_ok = True
            print(f'Terrain grid: {TERRAIN_GRID_N}x{TERRAIN_GRID_N}  '
                  f'z range [{_ZZ.min():.1f}, {_ZZ.max():.1f}] m (relative to anchor)')
        except ImportError as _e:
            print(f'  rasterio/scipy not available ({_e}) -- pip install rasterio scipy')
        except Exception as _e:
            print(f'  DEM processing failed: {_e}')

print(f'DEM terrain applied: {_dem_ok}')


## CELL 2c — OSM Roads, Vegetation, Water (real-world clutter)
USD scenes usually only contain the objects an artist explicitly modeled --
this fills the gaps around the USD buildings with real OpenStreetMap roads,
vegetation, and water for the same bbox (same tag patterns as
`sionna019_scene_builder_london.ipynb` CELL 4), draped onto the same DEM
terrain as the buildings in CELL 2b.

Roads -> buffered by per-highway-type half-width -> `itu_asphalt`.
Vegetation (forest/orchard/wood/scrub) -> extruded to canopy height -> `itu_vegetation`.
Water (natural=water) -> thin extruded polygon -> `itu_water`.

All produced meshes are appended to `usd_meshes`, so CELL 3 (material
mapping) and CELL 4/5 (PLY + scene.xml writers) pick them up automatically
-- no changes needed downstream.

Skips gracefully if `osmnx`/`shapely` aren't installed, or if
`USD_HAS_GEOREFERENCE=False` (no real-world bbox to query OSM against).


In [ ]:
# ============================================================
# CELL 2c — OSM ROADS + VEGETATION + WATER (REAL-WORLD CLUTTER)
# ============================================================
_osm_clutter_added = 0

if not USD_HAS_GEOREFERENCE:
    print('Skipping OSM clutter -- USD_HAS_GEOREFERENCE=False (no real-world bbox).')
else:
    try:
        import osmnx as ox
        ox.settings.use_cache = True
        ox.settings.log_console = False
        ox.settings.requests_timeout = 30  # fail fast instead of hanging on restricted networks
        _HAS_OSMNX = True
    except ImportError as _e:
        _HAS_OSMNX = False
        print(f'osmnx not available ({_e}) -- pip install osmnx. Skipping OSM clutter.')

    if _HAS_OSMNX:
        import numpy as np
        from pyproj import Transformer
        import shapely.geometry as sg

        _wgs_to_bng2 = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
        _bng_to_wgs2 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
        _anchor_e2, _anchor_n2 = _wgs_to_bng2.transform(ANCHOR_LON, ANCHOR_LAT)

        _all_xy2 = np.concatenate([m['verts'][:, :2] for m in usd_meshes
                                    if m['name'] != 'dem_terrain'], axis=0)
        _x0, _x1 = float(_all_xy2[:, 0].min()), float(_all_xy2[:, 0].max())
        _y0, _y1 = float(_all_xy2[:, 1].min()), float(_all_xy2[:, 1].max())
        if globals().get('USE_FIXED_SCENE_BBOX', False):
            # Same fixed area as sionna019_scene_builder_london.ipynb's SCENE_WEST/
            # EAST/SOUTH/NORTH, instead of just padding the USD building footprint.
            _w_lon, _s_lat, _e_lon, _n_lat = SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH
        else:
            _e0 = _anchor_e2 + _x0 - OSM_BBOX_PAD_M
            _e1 = _anchor_e2 + _x1 + OSM_BBOX_PAD_M
            _n0 = _anchor_n2 + _y0 - OSM_BBOX_PAD_M
            _n1 = _anchor_n2 + _y1 + OSM_BBOX_PAD_M
            _w_lon, _s_lat = _bng_to_wgs2.transform(_e0, _n0)
            _e_lon, _n_lat = _bng_to_wgs2.transform(_e1, _n1)
        print(f'OSM query bbox: lon=[{_w_lon:.5f},{_e_lon:.5f}]  lat=[{_s_lat:.5f},{_n_lat:.5f}]')

        import osmnx as _ox_ver
        _ox_version = tuple(int(x) for x in _ox_ver.__version__.split('.')[:2])

        def _features_from_bbox(tags):
            if _ox_version >= (2, 0):
                return ox.features_from_bbox(bbox=(_w_lon, _s_lat, _e_lon, _n_lat), tags=tags)
            elif _ox_version >= (1, 3):
                return ox.features_from_bbox(bbox=(_n_lat, _s_lat, _e_lon, _w_lon), tags=tags)
            else:
                return ox.features_from_bbox(north=_n_lat, south=_s_lat, east=_e_lon, west=_w_lon, tags=tags)

        def _to_local(lon, lat):
            e, n = _wgs_to_bng2.transform(lon, lat)
            return e - _anchor_e2, n - _anchor_n2

        def _fan_triangulate_ring(coords2d):
            """Simple centroid-fan triangulation. Exact for convex rings;
            an approximation for concave OSM footprints -- good enough for
            flat radio-material clutter, not a CAD-accurate mesh."""
            coords2d = np.asarray(coords2d, dtype=np.float64)
            if len(coords2d) >= 2 and np.allclose(coords2d[0], coords2d[-1]):
                coords2d = coords2d[:-1]
            return coords2d

        def _extrude_ring(coords2d, z0, height):
            ring = _fan_triangulate_ring(coords2d)
            m = len(ring)
            if m < 3:
                return None, None
            cx, cy = ring[:, 0].mean(), ring[:, 1].mean()
            bottom = np.column_stack([ring, np.full(m, z0)])
            top    = np.column_stack([ring, np.full(m, z0 + height)])
            cb = np.array([[cx, cy, z0]])
            ct = np.array([[cx, cy, z0 + height]])
            V = np.vstack([bottom, top, cb, ct]).astype(np.float32)
            cb_idx, ct_idx = 2 * m, 2 * m + 1
            F = []
            for i in range(m):
                j = (i + 1) % m
                F.append([cb_idx, j, i])                  # bottom fan (down-facing)
                F.append([ct_idx, m + i, m + j])           # top fan (up-facing)
                F.append([i, j, m + i])                     # wall tri 1
                F.append([j, m + j, m + i])                  # wall tri 2
            return V, np.asarray(F, dtype=np.int32)

        def _local_z_safe(x, y):
            if 'local_z' in dir():
                try:
                    return local_z(x, y)
                except Exception:
                    return 0.0
            return 0.0

        def _append_clutter_polygon(poly, height, name_prefix, material, idx):
            if poly.is_empty:
                return 0
            polys = list(poly.geoms) if isinstance(poly, sg.MultiPolygon) else [poly]
            n_added = 0
            for p in polys:
                if p.is_empty or p.area <= 0:
                    continue
                coords_local = [_to_local(lon, lat) for lon, lat in p.exterior.coords]
                V2, F2 = _extrude_ring(coords_local, 0.0, height)
                if V2 is None:
                    continue
                cx, cy = V2[:, 0].mean(), V2[:, 1].mean()
                dz = _local_z_safe(cx, cy)
                V2[:, 2] += dz
                usd_meshes.append({'name': f'{name_prefix}_{idx}_{n_added}',
                                    'verts': V2, 'faces': F2, 'material': material})
                n_added += 1
            return n_added

        # ── Roads ──────────────────────────────────────────────────────────
        if INCLUDE_OSM_ROADS:
            try:
                gdf_roads = _features_from_bbox({'highway': True})
                _ROAD_WIDTH = {'motorway': 12, 'trunk': 10, 'primary': 8, 'secondary': 7,
                               'tertiary': 6, 'residential': 5, 'unclassified': 4,
                               'service': 3, 'road': 4}
                _n_roads = 0
                for _i, (_, row) in enumerate(gdf_roads.iterrows()):
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    hw = str(row.get('highway', '')).lower()
                    width = _ROAD_WIDTH.get(hw, 4) / 2.0
                    lines = geom.geoms if isinstance(geom, sg.MultiLineString) else [geom]
                    for line in lines:
                        if not isinstance(line, sg.LineString):
                            continue
                        coords_local = [_to_local(lon, lat) for lon, lat in line.coords]
                        buffered = sg.LineString(coords_local).buffer(width, cap_style=2, join_style=2)
                        _n_roads += _append_clutter_polygon(buffered, ROAD_HEIGHT_M, 'osm_road', 'OSM_Road', _i)
                print(f'  Roads     : {_n_roads} segment mesh(es)')
                _osm_clutter_added += _n_roads
            except Exception as _e:
                print(f'  Roads download/build failed: {_e}')
        else:
            print('  Roads     : skipped (INCLUDE_OSM_ROADS=False)')

        # ── Vegetation ───────────────────────────────────────────────────────
        if INCLUDE_OSM_VEGETATION:
            try:
                gdf_veg = _features_from_bbox({'landuse': ['forest', 'orchard'],
                                                'natural': ['wood', 'scrub']})
                _n_veg = 0
                for _i, (_, row) in enumerate(gdf_veg.iterrows()):
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _n_veg += _append_clutter_polygon(geom, VEGETATION_HEIGHT_M, 'osm_vegetation', 'OSM_Vegetation', _i)
                print(f'  Vegetation: {_n_veg} polygon mesh(es)')
                _osm_clutter_added += _n_veg
            except Exception as _e:
                print(f'  Vegetation download/build failed: {_e}')
        else:
            print('  Vegetation: skipped (INCLUDE_OSM_VEGETATION=False)')

        # ── Water ──────────────────────────────────────────────────────────
        if INCLUDE_OSM_WATER:
            try:
                gdf_water = _features_from_bbox({'natural': 'water'})
                _n_water = 0
                for _i, (_, row) in enumerate(gdf_water.iterrows()):
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _n_water += _append_clutter_polygon(geom, WATER_HEIGHT_M, 'osm_water', 'OSM_Water', _i)
                print(f'  Water     : {_n_water} polygon mesh(es)')
                _osm_clutter_added += _n_water
            except Exception as _e:
                print(f'  Water download/build failed: {_e}')
        else:
            print('  Water     : skipped (INCLUDE_OSM_WATER=False)')

print(f'OSM clutter meshes added: {_osm_clutter_added}  (total usd_meshes: {len(usd_meshes)})')


## CELL 3 — Map USD Materials → ITU-R P.2040-2 Material Names
USD/Omniverse materials carry visual (PBR/MaterialX) names, not RF electromagnetic
properties — there is no built-in "this is concrete for radio purposes" semantic.
This cell matches each USD material's **name** against the same keyword set the
OSM pipeline uses, falling back to a default if nothing matches. If your USD
materials use opaque names (e.g. `Material_007`), you'll need to either rename
them before export or extend `_USD_MAT_KEYWORDS` with project-specific aliases —
matching by `diffuseColor`/MaterialX graph instead of name is a possible future
upgrade, not implemented here.

In [ ]:
# ============================================================
# CELL 3 — USD MATERIAL NAME → ITU-R MATERIAL MAPPING
# ============================================================
_USD_MAT_KEYWORDS = {
    'itu_concrete'  : ('concrete', 'cement', 'paving'),
    'itu_brick'     : ('brick',),
    'itu_glass'     : ('glass', 'window', 'curtain_wall', 'curtainwall'),
    'itu_metal'     : ('metal', 'steel', 'aluminium', 'aluminum', 'cladding'),
    'itu_wood'      : ('wood', 'timber', 'plywood'),
    'itu_plasterboard': ('plaster', 'drywall', 'gypsum'),
    'itu_marble'    : ('marble',),
    'itu_asphalt'   : ('asphalt', 'tarmac', 'road'),
    'itu_vegetation': ('vegetation', 'tree', 'foliage', 'grass', 'leaf'),
    'itu_water'     : ('water', 'pond', 'river', 'lake'),
    'itu_wet_ground': ('ground', 'soil', 'dirt', 'terrain'),
}
_DEFAULT_ITU_MAT = 'itu_concrete'

def _match_usd_material(usd_mat_name):
    n = usd_mat_name.lower()
    for itu_name, kws in _USD_MAT_KEYWORDS.items():
        if any(kw in n for kw in kws):
            return itu_name
    return None

print('USD material -> ITU mapping:')
mesh_itu_material = {}
for m in usd_meshes:
    matched = _match_usd_material(m['material'])
    itu_name = matched or _DEFAULT_ITU_MAT
    mesh_itu_material[m['name']] = itu_name
    flag = '' if matched else '  (no keyword match -- using default)'
    print(f"  {m['material']:<24} -> {itu_name:<16}{flag}")


## CELL 4 — Write Per-Mesh PLYs
Same writer (`_write_ply`) the OSM builder's CELL 4 uses, so downstream tooling
(CELL 5's `scene.xml` writer, `blender_to_sionna2_converter.py`, the simulation
notebooks) sees identical PLY files regardless of where the geometry came from.

In [ ]:
# ============================================================
# CELL 4 — WRITE PLYs
# ============================================================
def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:  f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces: f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

# mat_plys: {itu_material_name: [(relative_ply_path, role), ...]} -- same shape
# the OSM builder's CELL 5 scene.xml writer expects.
mat_plys = {}
ground_ply = None

for m in usd_meshes:
    itu_name = mesh_itu_material[m['name']]
    fname = f"usd_{m['name']}.ply"
    fpath = os.path.join(MESH_DIR, fname)
    _write_ply(m['verts'], m['faces'], fpath)
    rel_path = os.path.basename(MESH_DIR) + '/' + fname

    # 'dem_terrain' = real EA LiDAR terrain built in CELL 2b; 'ground' = the
    # flat synthetic ground slab (demo mode, or no DEM terrain available).
    is_ground = m['name'] in ('ground', 'dem_terrain')
    if is_ground:
        ground_ply = rel_path
        continue
    mat_plys.setdefault(itu_name, []).append((rel_path, 'usd_import'))

print(f'Wrote {len(usd_meshes)} PLY(s) -> {MESH_DIR}')
print(f'Ground PLY : {ground_ply or "(none found -- CELL 5 will fall back to a flat terrain.ply)"}')
print(f'mat_plys   : {{ {", ".join(f"{k}: {len(v)}" for k, v in mat_plys.items())} }}')

# CELL 5 (ported from the OSM builder) writes a fixed "terrain.ply" shape --
# reuse whichever ground/terrain mesh CELL 2/2b produced so the scene doesn't
# end up with two overlapping ground planes.
import shutil
terrain_ply_path = os.path.join(MESH_DIR, 'terrain.ply')
if ground_ply is not None:
    shutil.copyfile(os.path.join(SCENE_DIR, ground_ply), terrain_ply_path)
    print(f'Copied USD/DEM ground mesh -> {terrain_ply_path}')
elif not os.path.exists(terrain_ply_path):
    _gv, _gf = _box_mesh(0, 0, 0, 1000, 1000, 0.0)
    _write_ply(_gv, _gf, terrain_ply_path)
    print(f'No ground mesh in USD scene -- wrote flat 1km x 1km placeholder terrain.ply')


## CELL 5 — Write scene.xml (Sionna 0.19 Mitsuba format)
Identical material-parameter table, colour table, and `_guard_bsdf_refs` safety
net as `sionna019_scene_builder_london.ipynb` CELL 5 — only the source of
`mat_plys` differs (USD meshes here, OSM footprints there), so the XML output
format is a guaranteed drop-in.

In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
_ITU_P2040_PARAMS = {
    #                    a       b       c        d      s     xpd
    'itu_concrete' : (5.31,  0.000,  0.0326, 0.8095, 0.30, 0.10),
    'itu_brick'    : (3.91,  0.000,  0.0238, 0.0000, 0.25, 0.10),
    'itu_glass'    : (6.27,  0.000,  0.0043, 1.1925, 0.10, 0.05),
    'itu_plywood'  : (1.99,  0.000,  0.0047, 1.0718, 0.20, 0.10),
    'itu_metal'    : (1.00,  0.000,  1.0e7,  0.0000, 0.05, 0.05),
    'itu_wet_ground'       : (30.0, -0.400, 0.1500, 1.3000, 0.35, 0.05),
    'itu_water'            : (80.0,  0.000, 0.0100, 0.0000, 0.02, 0.05),
    'itu_medium_dry_ground': (15.0, -0.100, 0.0350, 1.6300, 0.10, 0.05),
    'itu_very_dry_ground'  : ( 3.0,  0.000, 0.00015,2.5200, 0.10, 0.05),
    'itu_vegetation'       : ( 1.50, 0.000, 0.0020, 0.5000, 0.40, 0.50),
    'itu_asphalt'          : ( 2.56, 0.000, 0.0050, 0.0000, 0.30, 0.15),
}
_f_ghz_c5 = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
ITU_MATERIALS = {}
for _mn, (_a, _b, _c, _d, _s, _xpd) in _ITU_P2040_PARAMS.items():
    ITU_MATERIALS[_mn] = (round(_a * (_f_ghz_c5 ** _b), 6), round(_c * (_f_ghz_c5 ** _d), 8), _s, _xpd)

_MAT_REMAP_019 = {
    'itu_wood'        : 'itu_plywood',
    'itu_water'       : 'itu_medium_dry_ground',
    'itu_vegetation'  : 'itu_ceiling_board',
    'itu_asphalt'     : 'itu_very_dry_ground',
    'itu_marble'      : 'itu_concrete',
    'itu_plasterboard': 'itu_ceiling_board',
}
TERRAIN_MATERIAL = 'itu_wet_ground'

_ITU_COLOURS_019 = {
    'itu_concrete'         : '0.539 0.539 0.539',
    'itu_brick'            : '1.000 0.498 0.055',
    'itu_glass'            : '0.596 0.875 0.541',
    'itu_plywood'          : '0.514 0.376 0.220',
    'itu_metal'            : '0.220 0.220 0.254',
    'itu_wet_ground'       : '0.910 0.569 0.055',
    'itu_very_dry_ground'  : '0.498 0.498 0.498',
    'itu_medium_dry_ground': '0.780 0.780 0.780',
    'itu_ceiling_board'    : '0.180 0.450 0.180',
}

used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}
_used_mats_remapped = {_MAT_REMAP_019.get(m, m) for m in used_mats} | {TERRAIN_MATERIAL}

lines = ['<?xml version="1.0" encoding="utf-8"?>', '<scene version="2.1.0">', '',
          '  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->']
for mat_name in sorted(_used_mats_remapped):
    rgb = _ITU_COLOURS_019.get(mat_name, '0.5 0.5 0.5')
    lines += [f'  <bsdf type="diffuse" id="{mat_name}">',
              f'    <rgb name="reflectance" value="{rgb}"/>',
              '  </bsdf>', '']

lines += ['  <!-- ── Terrain ─────────────────────────────────────── -->',
          '  <shape type="ply" id="mesh-ground">',
          '    <string name="filename" value="meshes/terrain.ply"/>',
          f'    <ref id="{TERRAIN_MATERIAL}" name="bsdf"/>',
          '    <boolean name="face_normals" value="true"/>',
          '  </shape>', '']

lines.append('  <!-- ── USD-imported geometry ───────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    ref_mat = _MAT_REMAP_019.get(mat_name, mat_name)
    for ply_path, role in ply_list:
        mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines += [f'  <shape type="ply" id="{mesh_id}">',
                  f'    <string name="filename" value="{ply_path}"/>',
                  f'    <ref id="{ref_mat}" name="bsdf"/>',
                  '    <boolean name="face_normals" value="true"/>',
                  '  </shape>']
lines += ['', '</scene>']

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes - 1} USD-imported parts)')

meta = {'utm_epsg': UTM_EPSG, 'projection_crs': PROJECTION_CRS,
        'source': 'usd_omniverse', 'demo_mode': DEMO_MODE,
        'n_meshes': len(usd_meshes)}
with open(os.path.join(BASE_DIR, 'scene_parameters.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {os.path.join(BASE_DIR, "scene_parameters.json")}')


## CELL 6 — 2D Top-Down Preview

In [ ]:
# ============================================================
# CELL 6 — 2D PREVIEW OF EXTRACTED GEOMETRY
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))
_cmap = plt.get_cmap('tab10')
_mat_colors = {m: _cmap(i % 10) for i, m in enumerate(sorted({mesh_itu_material[m['name']] for m in usd_meshes}))}

for m in usd_meshes:
    v = m['verts']
    xs, ys = v[:, 0], v[:, 1]
    if m['name'] == 'dem_terrain':
        # Dense N x N grid mesh -- angle-sort-and-fill produces a starburst
        # of crossing lines on a grid (it only works for small convex
        # footprints like the building boxes below). Draw it as a flat
        # extent rectangle + a light hatch instead, so it doesn't swamp
        # the actual buildings.
        ax.add_patch(mpatches.Rectangle(
            (xs.min(), ys.min()), xs.max() - xs.min(), ys.max() - ys.min(),
            facecolor=_mat_colors[mesh_itu_material[m['name']]], alpha=0.15,
            edgecolor='none', zorder=0))
        continue
    hull_order = np.argsort(np.arctan2(ys - ys.mean(), xs - xs.mean()))
    ax.fill(xs[hull_order], ys[hull_order], alpha=0.5,
            color=_mat_colors[mesh_itu_material[m['name']]], edgecolor='k', linewidth=0.5,
            zorder=2)
    ax.text(xs.mean(), ys.mean(), m['name'], fontsize=7, ha='center', zorder=3)

handles = [mpatches.Patch(color=c, label=m) for m, c in _mat_colors.items()]
ax.legend(handles=handles, loc='upper right', fontsize=8)
ax.set_xlabel('Local X (m)'); ax.set_ylabel('Local Y (m)')
ax.set_title(f'USD scene preview ({"DEMO" if DEMO_MODE else USD_FILE})  --  {len(usd_meshes)} prims')
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout()
_png = os.path.join(SCENE_DIR, 'usd_scene_preview.png')
plt.savefig(_png, dpi=130)
plt.close()
try:
    from IPython.display import Image, display
    display(Image(filename=_png, width=700))
except Exception:
    pass
print(f'Saved preview -> {_png}')


## CELL 7 — Load into Sionna RT and Sanity-Check
Requires Sionna to be installed; safe to skip if you only want to inspect the
written `scene.xml`/PLYs.

In [ ]:
# ============================================================
# CELL 7 — LOAD INTO SIONNA RT
# ============================================================
try:
    import sionna
    from sionna.rt import load_scene
    print(f'Sionna : {sionna.__version__}')
    scene = load_scene(scene_xml)
    print(f'Loaded : {scene_xml}')
    print(f'Radio materials ({len(scene.radio_materials)}): {sorted(scene.radio_materials.keys())}')
    print(f'Scene objects ({len(scene.objects)}): {sorted(scene.objects.keys())}')
    bbox = scene.mi_scene.bbox()
    print(f'BBox   : min={list(bbox.min)}  max={list(bbox.max)}')
except ImportError as _e:
    print(f'Sionna not installed in this environment -- skipping load check ({_e})')
except Exception as _e:
    print(f'Scene load FAILED: {_e}')
    import traceback; traceback.print_exc()
